In [3]:
import pandas as pd


In [12]:
# load dataset
df = pd.read_csv("dataset/Sample - Superstore.csv", encoding="windows-1252")
print("Rows and Columns loaded:", df.shape)
print(df.head())

Rows and Columns loaded: (9994, 21)
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category 

In [13]:
# 1. aggregate sales by category and product name
product_sales = (
    df.groupby(['Category', 'Product Name'])['Sales']
    .sum()
    .reset_index()
)

product_sales

,Category,Product Name,Sales
0,Furniture,12-1/2 Diameter Round Wall Clock,551.448
1,Furniture,24-Hour Round Wall Clock,487.512
2,Furniture,36X48 HARDFLOOR CHAIRMAT,369.248
3,Furniture,3M Hangers With Command Adhesive,108.040
4,Furniture,3M Polarizing Light Filter Sleeves,216.340
...,...,...,...
1845,Technology,iKross Bluetooth Portable Keyboard + Cell Phon...,477.660
1846,Technology,iOttie HLCRIO102 Car Mount,215.892
1847,Technology,iOttie XL Car Mount,223.888
1848,Technology,invisibleSHIELD by ZAGG Smudge-Free Screen Pro...,442.554


In [14]:
#2. Critical step: Explicitly sort by Category and Sales (Descending)
product_sales = product_sales.sort_values(
    ['Category', 'Sales'],
    ascending= [True, False]
)

product_sales

,Category,Product Name,Sales
206,Furniture,HON 5400 Series Task Chairs for Big and Tall,21870.5760
321,Furniture,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.9656
40,Furniture,Bretford Rectangular Conference Table Tops,12995.2915
201,Furniture,Global Troy Executive Leather Low-Back Tilter,12975.3820
325,Furniture,SAFCO Arco Folding Chair,11572.7800
...,...,...,...
1660,Technology,Maxell CD-R Discs,15.7600
1793,Technology,Sannysis Cute Owl Design Soft Skin Case Cover ...,13.8600
1676,Technology,Memorex Mini Travel Drive 4 GB USB 2.0 Flash D...,13.6160
1653,Technology,Maxell 4.7GB DVD+R 5/Pack,11.6820


In [15]:
# 3\. Calculate rank within each category partition (equivalent to ROW\_NUMBER() OVER PARTITION BY)
product_sales['sales_rank'] = product_sales.groupby('Category')['Sales'].rank(
    method = 'first',
    ascending= False
)
product_sales

,Category,Product Name,Sales,sales_rank
206,Furniture,HON 5400 Series Task Chairs for Big and Tall,21870.5760,1.0
321,Furniture,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.9656,2.0
40,Furniture,Bretford Rectangular Conference Table Tops,12995.2915,3.0
201,Furniture,Global Troy Executive Leather Low-Back Tilter,12975.3820,4.0
325,Furniture,SAFCO Arco Folding Chair,11572.7800,5.0
...,...,...,...,...
1660,Technology,Maxell CD-R Discs,15.7600,408.0
1793,Technology,Sannysis Cute Owl Design Soft Skin Case Cover ...,13.8600,409.0
1676,Technology,Memorex Mini Travel Drive 4 GB USB 2.0 Flash D...,13.6160,410.0
1653,Technology,Maxell 4.7GB DVD+R 5/Pack,11.6820,411.0


In [16]:
# When doing this in Python, **never use** **df.apply(lambda row: ...)** in senior tech interviews—it
# iterates line-by-line in pure Python at \~1,000 rows/sec. Instead, use **np.select()**,
# which evaluates C-compiled memory blocks at 1,000,000+ rows/sec:

import numpy as np

# Define vectorized boolean conditions
conditions = [
    df['Profit'] <0,
    (df['Profit'] >= 0) & (df['Profit'] <=100)
]
choices = ['LOSS', 'MODERATE']

# vectorized assignment matching SQL's CASE WHEN
df['profit_tier'] = np.select(conditions, choices, default= 'HIGHT PROFIT')

print(df[['Order ID', 'Profit', 'profit_tier']].head(15))

          Order ID    Profit   profit_tier
0   CA-2016-152156   41.9136      MODERATE
1   CA-2016-152156  219.5820  HIGHT PROFIT
2   CA-2016-138688    6.8714      MODERATE
3   US-2015-108966 -383.0310          LOSS
4   US-2015-108966    2.5164      MODERATE
5   CA-2014-115812   14.1694      MODERATE
6   CA-2014-115812    1.9656      MODERATE
7   CA-2014-115812   90.7152      MODERATE
8   CA-2014-115812    5.7825      MODERATE
9   CA-2014-115812   34.4700      MODERATE
10  CA-2014-115812   85.3092      MODERATE
11  CA-2014-115812   68.3568      MODERATE
12  CA-2017-114412    5.4432      MODERATE
13  CA-2016-161389  132.5922  HIGHT PROFIT
14  US-2015-118983 -123.8580          LOSS
